In [1]:
import pandas as pd

In [48]:
df = pd.read_csv('StudentsPerformance.csv')

In [49]:
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [50]:
len(df)

1000

#### Putting the data into PostgreSQL

In [51]:
# Checking data information

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   gender                       1000 non-null   str  
 1   race/ethnicity               1000 non-null   str  
 2   parental level of education  1000 non-null   str  
 3   lunch                        1000 non-null   str  
 4   test preparation course      1000 non-null   str  
 5   math score                   1000 non-null   int64
 6   reading score                1000 non-null   int64
 7   writing score                1000 non-null   int64
dtypes: int64(3), str(5)
memory usage: 103.5 KB


In [52]:
# Inserting student id column

df.insert(
    loc=0,
    column="student_id",
    value=[f"S{str(i).zfill(4)}" for i in range(1, len(df) +1)]
)

In [53]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   student_id                   1000 non-null   str  
 1   gender                       1000 non-null   str  
 2   race/ethnicity               1000 non-null   str  
 3   parental level of education  1000 non-null   str  
 4   lunch                        1000 non-null   str  
 5   test preparation course      1000 non-null   str  
 6   math score                   1000 non-null   int64
 7   reading score                1000 non-null   int64
 8   writing score                1000 non-null   int64
dtypes: int64(3), str(6)
memory usage: 116.2 KB


In [54]:
df.head()

,student_id,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,S0001,female,group B,bachelor's degree,standard,none,72,72,74
1,S0002,female,group C,some college,standard,completed,69,90,88
2,S0003,female,group B,master's degree,standard,none,90,95,93
3,S0004,male,group A,associate's degree,free/reduced,none,47,57,44
4,S0005,male,group C,some college,standard,none,76,78,75


In [55]:
# Adding datetime column

base_datetime = pd.to_datetime("2026-01-28 08:00:00")

df["datetime"] = [
    base_datetime +
    pd.Timedelta(seconds=i)
       for i in range(len(df))
]

In [56]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   student_id                   1000 non-null   str           
 1   gender                       1000 non-null   str           
 2   race/ethnicity               1000 non-null   str           
 3   parental level of education  1000 non-null   str           
 4   lunch                        1000 non-null   str           
 5   test preparation course      1000 non-null   str           
 6   math score                   1000 non-null   int64         
 7   reading score                1000 non-null   int64         
 8   writing score                1000 non-null   int64         
 9   datetime                     1000 non-null   datetime64[us]
dtypes: datetime64[us](1), int64(3), str(6)
memory usage: 124.0 KB


In [57]:
# Removing date time column as last column

datetime_col = df.pop("datetime")
df.insert(1, "datetime", datetime_col)

In [58]:
df.head()

,student_id,datetime,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,S0001,2026-01-28 08:00:00,female,group B,bachelor's degree,standard,none,72,72,74
1,S0002,2026-01-28 08:00:01,female,group C,some college,standard,completed,69,90,88
2,S0003,2026-01-28 08:00:02,female,group B,master's degree,standard,none,90,95,93
3,S0004,2026-01-28 08:00:03,male,group A,associate's degree,free/reduced,none,47,57,44
4,S0005,2026-01-28 08:00:04,male,group C,some college,standard,none,76,78,75


In [60]:
df.to_csv("student_performance.csv", index=False)

In [62]:
import psycopg2
import pandas as pd
from psycopg2.extras import execute_values
import numpy

df = pd.read_csv("student_performance.csv")

df.columns = [
    "student_id",
    "datetime",
    "gender",
    "race/ethnicity",
    "parental level of education",
    "lunch",
    "test preparation course",
    "math score",
    "reading score",
    "writing score"
]

conn = psycopg2.connect(
    host="localhost",
    port=5433,
    database="ng_education",
    user="postgres",
    password="anuoluwapo"
)

print("psycopg2 connected successfully")
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS
student_performance (
     student_id                  TEXT PRIMARY KEY,
     datetime                    TIMESTAMP,
     gender                      TEXT,
     race_ethnicity              TEXT,
     parental_level_of_education TEXT,
     lunch                       TEXT,
     test_preparation_course     TEXT,
     math_score                  INTEGER,
     reading_score               INTEGER,
     writing_score               INTEGER
);
""")


records = df.to_numpy().tolist()

insert_sql = """
INSERT INTO student_performance (
    student_id, datetime,
    gender, race_ethnicity, parental_level_of_education,
    lunch, test_preparation_course, math_score, reading_score, writing_score
)
VALUES %s
"""

execute_values(cur, insert_sql, records)

conn.commit()
cur.close()
conn.close()

print("Data loaded successfully")

psycopg2 connected successfully
Data loaded successfully
